In [1]:
import torch
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, CrossEntropyLoss
from torch.utils.data import DataLoader
from torch.optim import Adam

In [2]:
dataset = load_iris()

X_treino, X_teste, y_treino, y_teste = train_test_split(dataset.data, dataset.target, test_size=0.2, random_state=42)

In [3]:
model = Sequential(
    Linear(4, 32),
    ReLU(),
    Linear(32,16),
    ReLU(),
    Linear(16, 3)
)

In [4]:
optmizer = Adam(model.parameters(), lr=0.001)
criterion = CrossEntropyLoss()
max_epochs = 500
batch = 8

In [5]:
total_batchs = len(X_treino) // batch
last_batch = len(X_treino) % batch
has_last_b = last_batch > 0

print(total_batchs)
print(last_batch)

15
0


In [6]:
def train_step_class(x, y):
    optmizer.zero_grad()
    output = model(x)
    loss = criterion(output, y)
    loss.backward()
    optmizer.step()

def train_model_class():
    for i_epochs in range(max_epochs):
        model.train()
        if has_last_b:
            batch_x = Tensor(X_treino[-last_batch:])
            batch_y = Tensor(y_treino[-last_batch:]).long()
            train_step_class(batch_x, batch_y)
            
        for i_batchs in range(total_batchs):
            batch_x = Tensor(X_treino[(i_batchs * batch):((i_batchs + 1) * batch)])
            batch_y = Tensor(y_treino[(i_batchs * batch):((i_batchs + 1) * batch)]).long()
            train_step_class(batch_x, batch_y)

def eval_model_class():
    model.eval()
    with torch.no_grad():
        X_teste_tensor = Tensor(X_teste)
        output = model(X_teste_tensor).cpu()
        result = torch.argmax(output, dim=1).cpu().numpy()
        print(f"Acc: {accuracy_score(y_teste, result)}")
        print(f"F1score: {f1_score(y_teste, result, average="macro")}")
        print(f"CM: {confusion_matrix(y_teste, result)}")


In [7]:
train_model_class()
eval_model_class()

Acc: 1.0
F1score: 1.0
CM: [[10  0  0]
 [ 0  9  0]
 [ 0  0 11]]
